# RUL and Basic Feature Engineering

This notebook creates the target variable (RUL) and basic sensor-based features for the NASA C-MAPSS FD001 training dataset.

In [1]:
import pandas as pd

columns = ["id", "cycle"] + [f"setting_{i}" for i in range(1, 4)] + [f"s{i}" for i in range(1, 22)]

df = pd.read_csv(
    "../data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

df.head()

,id,cycle,setting_1,setting_2,setting_3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## RUL Construction

RUL (Remaining Useful Life) represents how many cycles are left until engine failure.
For the training data, each engine runs until failure, so RUL can be calculated from the maximum cycle of each engine.

In [2]:
df["max_cycle"] = df.groupby("id")["cycle"].transform("max")
df["RUL"] = df["max_cycle"] - df["cycle"]

df[["id", "cycle", "max_cycle", "RUL"]].head()

,id,cycle,max_cycle,RUL
0,1,1,192,191
1,1,2,192,190
2,1,3,192,189
3,1,4,192,188
4,1,5,192,187


## Basic Feature Engineering

In this section, we create simple features to better understand engine behavior over time.

### Sensor Baseline: Mean per Engine

`s2_mean_per_engine` represents the average value of sensor `s2` for each engine.
This gives us a simple baseline for normal behavior.

In [3]:
df["s2_mean_per_engine"] = df.groupby("id")["s2"].transform("mean")

df[["id", "cycle", "s2", "s2_mean_per_engine"]].head()

,id,cycle,s2,s2_mean_per_engine
0,1,1,641.82,642.621042
1,1,2,642.15,642.621042
2,1,3,642.35,642.621042
3,1,4,642.35,642.621042
4,1,5,642.37,642.621042


### Sensor Deviation from Baseline

`s2_diff` measures how far the current sensor value is from the engine's average behavior.
This can act as a simple anomaly signal.

In [4]:
df["s2_diff"] = df["s2"] - df["s2_mean_per_engine"]

df[["id", "cycle", "s2", "s2_mean_per_engine", "s2_diff"]].head()

,id,cycle,s2,s2_mean_per_engine,s2_diff
0,1,1,641.82,642.621042,-0.801042
1,1,2,642.15,642.621042,-0.471042
2,1,3,642.35,642.621042,-0.271042
3,1,4,642.35,642.621042,-0.271042
4,1,5,642.37,642.621042,-0.251042


### Cycle Progress

`cycle_progress` shows how much of the engine's observed lifetime has been completed.
It scales each engine's cycle from 0 to 1.

In [5]:
df["cycle_progress"] = df["cycle"] / df["max_cycle"]

df[["id", "cycle", "max_cycle", "cycle_progress"]].head()

,id,cycle,max_cycle,cycle_progress
0,1,1,192,0.005208
1,1,2,192,0.010417
2,1,3,192,0.015625
3,1,4,192,0.020833
4,1,5,192,0.026042


### Sensor Change from Previous Cycle

`s2_change` measures how much sensor `s2` changed compared to the previous cycle of the same engine.
This can help detect sudden jumps or abnormal changes.

In [6]:
df["s2_change"] = df.groupby("id")["s2"].transform(
    lambda x: x - x.shift(1)
)

df[["id", "cycle", "s2", "s2_change"]].head(10)

,id,cycle,s2,s2_change
0,1,1,641.82,NaN
1,1,2,642.15,0.33
2,1,3,642.35,0.20
3,1,4,642.35,0.00
4,1,5,642.37,0.02
5,1,6,642.10,-0.27
6,1,7,642.48,0.38
7,1,8,642.56,0.08
8,1,9,642.12,-0.44
9,1,10,641.71,-0.41


## Summary

In this notebook, we created:

- `RUL`: target variable for remaining useful life prediction
- `s2_mean_per_engine`: baseline behavior of sensor `s2`
- `s2_diff`: deviation from baseline
- `cycle_progress`: normalized lifetime progress
- `s2_change`: cycle-to-cycle sensor change

These features are exploratory and will be evaluated later before being used in modeling.

In [7]:
df[[
    "id",
    "cycle",
    "max_cycle",
    "RUL",
    "s2_mean_per_engine",
    "s2_diff",
    "cycle_progress",
    "s2_change"
]].head(10)

,id,cycle,max_cycle,RUL,s2_mean_per_engine,s2_diff,cycle_progress,s2_change
0,1,1,192,191,642.621042,-0.801042,0.005208,NaN
1,1,2,192,190,642.621042,-0.471042,0.010417,0.33
2,1,3,192,189,642.621042,-0.271042,0.015625,0.20
3,1,4,192,188,642.621042,-0.271042,0.020833,0.00
4,1,5,192,187,642.621042,-0.251042,0.026042,0.02
5,1,6,192,186,642.621042,-0.521042,0.031250,-0.27
6,1,7,192,185,642.621042,-0.141042,0.036458,0.38
7,1,8,192,184,642.621042,-0.061042,0.041667,0.08
8,1,9,192,183,642.621042,-0.501042,0.046875,-0.44
9,1,10,192,182,642.621042,-0.911042,0.052083,-0.41


In [8]:
df[[
    "max_cycle",
    "RUL",
    "s2_mean_per_engine",
    "s2_diff",
    "cycle_progress",
    "s2_change"
]].isna().sum()

max_cycle               0
RUL                     0
s2_mean_per_engine      0
s2_diff                 0
cycle_progress          0
s2_change             100
dtype: int64